# Week 4 Lab: Advanced SQL, Views, Triggers, and Transactions

## Lab Overview
In this session you will:
1. Provision a dedicated schema for experimentation.
2. Create tables with robust constraints to protect data quality.
3. Build a reporting view and examine materialized view refresh options.
4. Author a trigger function that records inventory changes.
5. Run controlled transactions to observe ACID behaviour.

By the end you should have SQL artifacts that can seed your midterm project proposal.

## Prerequisites
- PostgreSQL 14+ (Supabase project or local instance)
- Service-role or superuser connection string with privileges to create schemas, roles, and extensions
- Python 3.10+ with `psycopg` available
- Access to the `pgcrypto` extension (`gen_random_uuid`) and permission to call `pg_sleep`

> **Tip:** Run this lab on a staging database. The setup drops and recreates the `week4_lab` schema.

In [ ]:
# Optional dependency installation if running in a fresh environment
# %pip install psycopg[binary] python-dotenv tabulate

## Configure Connection
Store the connection string as an environment variable before running subsequent cells. For Supabase, paste the service-role connection string (from the SQL editor connection panel).

In [ ]:
import os
from getpass import getpass

if 'WEEK4_DB_URL' not in os.environ or not os.environ['WEEK4_DB_URL']:
    os.environ['WEEK4_DB_URL'] = getpass('Enter PostgreSQL connection string: ')

DATABASE_URL = os.environ['WEEK4_DB_URL']
print('Using database:', DATABASE_URL.split('@')[-1])

In [ ]:
import psycopg
from psycopg.rows import dict_row

conn = psycopg.connect(DATABASE_URL, autocommit=True)


def run(sql: str, params=None):
    "Execute SQL and optionally return rows."
    with conn.cursor(row_factory=dict_row) as cur:
        cur.execute(sql, params or {})
        if cur.description:
            return cur.fetchall()

print('Connection established.')

## Schema Reset
Ensure a clean environment by removing prior lab artifacts and creating the working schema.

In [ ]:
run('DROP SCHEMA IF EXISTS week4_lab CASCADE;')
run('CREATE SCHEMA week4_lab;')
run('SET search_path TO week4_lab;')
run('CREATE EXTENSION IF NOT EXISTS pgcrypto;')
print('Schema recreated and extension enabled.')

## Base Tables & Sample Data
The dataset models a simple commerce workflow. Execute the cell to create tables and seed sample data that will power the remaining exercises.

In [ ]:
run('''
    SET search_path = week4_lab;

    CREATE TABLE products (
        id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
        sku text UNIQUE NOT NULL,
        name text NOT NULL,
        unit_price numeric(12,2) NOT NULL CHECK (unit_price >= 0)
    );

    CREATE TABLE customers (
        id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
        email text UNIQUE NOT NULL,
        full_name text NOT NULL,
        created_at timestamptz NOT NULL DEFAULT now()
    );

    CREATE TABLE orders (
        id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
        customer_id uuid NOT NULL REFERENCES customers(id) ON DELETE RESTRICT,
        status text NOT NULL DEFAULT 'pending' CHECK (status IN ('pending','paid','cancelled','shipped')),
        order_total numeric(12,2) NOT NULL DEFAULT 0 CHECK (order_total >= 0),
        created_at timestamptz NOT NULL DEFAULT now(),
        updated_at timestamptz NOT NULL DEFAULT now()
    );

    CREATE TABLE order_items (
        id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
        order_id uuid NOT NULL REFERENCES orders(id) ON DELETE CASCADE,
        product_id uuid NOT NULL REFERENCES products(id) ON DELETE RESTRICT,
        quantity integer NOT NULL CHECK (quantity > 0),
        unit_price numeric(12,2) NOT NULL CHECK (unit_price >= 0)
    );

    CREATE TABLE inventory_ledger (
        id bigserial PRIMARY KEY,
        product_id uuid NOT NULL REFERENCES products(id),
        change_type text NOT NULL CHECK (change_type IN ('sale','restock','adjustment')),
        delta integer NOT NULL,
        balance_after integer NOT NULL,
        occurred_at timestamptz NOT NULL DEFAULT now(),
        details jsonb NOT NULL DEFAULT '{}'::jsonb
    );
''')

run('''
    INSERT INTO products (sku, name, unit_price)
    VALUES
        ('SKU-100', 'Cloud Hoodie', 79.00),
        ('SKU-200', 'Database Sticker Pack', 5.00),
        ('SKU-300', 'Query Optimization Guide', 39.50);

    INSERT INTO customers (email, full_name)
    VALUES
        ('lee@dataclub.io', 'Lee Wong'),
        ('maya@dataclub.io', 'Maya Hernandez');

    INSERT INTO orders (customer_id, status, order_total)
    SELECT id, 'paid', 118.00 FROM customers WHERE email = 'lee@dataclub.io';

    WITH oi AS (
        SELECT o.id AS order_id,
               p1.id AS hoodie,
               p2.id AS stickers
        FROM orders o
        JOIN customers c ON o.customer_id = c.id
        JOIN products p1 ON p1.sku = 'SKU-100'
        JOIN products p2 ON p2.sku = 'SKU-200'
        WHERE c.email = 'lee@dataclub.io'
    )
    INSERT INTO order_items (order_id, product_id, quantity, unit_price)
    SELECT order_id, hoodie, 1, 79.00 FROM oi
    UNION ALL
    SELECT order_id, stickers, 4, 5.00 FROM oi;

    INSERT INTO inventory_ledger (product_id, change_type, delta, balance_after, details)
    SELECT id, 'restock', 50, 50, jsonb_build_object('note','Initial stock') FROM products;
''')

print('Tables populated with starter dataset.')

## Task 1 — Build a Reporting View
Create `week4_lab.vw_order_summary` that exposes order metadata and aggregated revenue. Run the cell to create the view and inspect the output.

In [ ]:
run('''
    CREATE OR REPLACE VIEW week4_lab.vw_order_summary AS
    SELECT
        o.id AS order_id,
        c.full_name,
        c.email,
        o.status,
        o.created_at,
        o.updated_at,
        COUNT(oi.id) AS line_items,
        SUM(oi.quantity * oi.unit_price)::numeric(12,2) AS calculated_total
    FROM week4_lab.orders o
    JOIN week4_lab.customers c ON c.id = o.customer_id
    JOIN week4_lab.order_items oi ON oi.order_id = o.id
    GROUP BY o.id, c.full_name, c.email, o.status, o.created_at, o.updated_at;
''')

run('SELECT * FROM week4_lab.vw_order_summary;')

### Instructor Note — Order Summary View
Expected output: one row per order with aggregated `line_items` and `calculated_total`. With seed data, the view returns a single row (Lee Wong) showing 2 line items and 99.00 revenue. Use this to confirm joins and aggregation are correct.

### Optional: Materialized View
Materialized views cache expensive aggregations. Create the view and perform a refresh after inserts.

In [ ]:
run('DROP MATERIALIZED VIEW IF EXISTS week4_lab.mv_revenue_by_product;')
run('''
    CREATE MATERIALIZED VIEW week4_lab.mv_revenue_by_product AS
    SELECT
        p.sku,
        p.name,
        SUM(oi.quantity) AS units_sold,
        SUM(oi.quantity * oi.unit_price)::numeric(12,2) AS revenue
    FROM week4_lab.order_items oi
    JOIN week4_lab.products p ON p.id = oi.product_id
    GROUP BY p.sku, p.name;
''')

run('REFRESH MATERIALIZED VIEW week4_lab.mv_revenue_by_product;')
run('SELECT * FROM week4_lab.mv_revenue_by_product;')

### Instructor Note — Materialized View
After creation and refresh, expect three rows (one per product) showing total units and revenue. Demonstrate how a new `order_items` insert requires another `REFRESH` to update figures.

## Task 2 — Enforce Advanced Constraints
Add a deferrable constraint that ensures each paid order has at least one item. The check executes at commit time, allowing staged inserts.

In [ ]:
run('ALTER TABLE week4_lab.orders DROP CONSTRAINT IF EXISTS orders_has_items;')
run('''
    ALTER TABLE week4_lab.orders
    ADD CONSTRAINT orders_has_items
    DEFERRABLE INITIALLY DEFERRED
    CHECK (
        status <> 'paid'
        OR EXISTS (
            SELECT 1 FROM week4_lab.order_items oi WHERE oi.order_id = id
        )
    );
''')

print('Deferrable constraint installed.')

### Instructor Note — Deferrable Constraint
This constraint fires at commit. Emphasise that it allows inserting order headers before items within the same transaction, but blocks committing a `paid` order without items. Point out structure for students to reuse in proposals.

### Constraint Validation Test
Attempt to mark a new order as paid without items. The transaction should raise a check violation on commit.

In [ ]:
with conn.cursor() as cur:
    cur.execute('BEGIN;')
    cur.execute('SET search_path TO week4_lab;')
    cur.execute('INSERT INTO orders (customer_id, status, order_total) VALUES ((SELECT id FROM customers LIMIT 1), ''paid'', 10);')
    try:
        cur.execute('COMMIT;')
    except psycopg.errors.CheckViolation as err:
        print('Expected violation:', err.diag.message_primary)
        cur.execute('ROLLBACK;')

### Instructor Note — Constraint Violation Demo
When run, PostgreSQL raises `ERROR: new row for relation "orders" violates check constraint "orders_has_items"`. Highlight deferrable behaviour by noting the error only appears on COMMIT.

## Task 3 — Trigger-Based Inventory Logging
Implement a trigger that records inventory consumption whenever an order item is inserted.

In [ ]:
run('DROP FUNCTION IF EXISTS week4_lab.log_inventory_change() CASCADE;')
run('''
    CREATE OR REPLACE FUNCTION week4_lab.log_inventory_change()
    RETURNS trigger
    LANGUAGE plpgsql
    SECURITY DEFINER
    SET search_path = week4_lab
    AS $$
    DECLARE
        current_balance integer;
    BEGIN
        SELECT COALESCE(balance_after, 0)
        INTO current_balance
        FROM week4_lab.inventory_ledger
        WHERE product_id = NEW.product_id
        ORDER BY occurred_at DESC, id DESC
        LIMIT 1;

        current_balance := current_balance - NEW.quantity;

        INSERT INTO week4_lab.inventory_ledger (product_id, change_type, delta, balance_after, details)
        VALUES (
            NEW.product_id,
            'sale',
            -NEW.quantity,
            current_balance,
            jsonb_build_object('order_id', NEW.order_id)
        );
        RETURN NEW;
    END;
    $$;
''')

run('DROP TRIGGER IF EXISTS trg_order_items_sale ON week4_lab.order_items;')
run('''
    CREATE TRIGGER trg_order_items_sale
    AFTER INSERT ON week4_lab.order_items
    FOR EACH ROW
    EXECUTE FUNCTION week4_lab.log_inventory_change();
''')

print('Trigger installed.')

### Instructor Note — Trigger Function
Review logic: fetch latest balance, subtract quantity, insert ledger row with negative delta. Mention SECURITY DEFINER requirement because trigger runs as table owner.

### Trigger Validation
Insert an additional item and inspect the ledger entries to verify the trigger executed.

In [ ]:
run('''
    INSERT INTO week4_lab.order_items (order_id, product_id, quantity, unit_price)
    SELECT o.id, p.id, 2, p.unit_price
    FROM week4_lab.orders o
    JOIN week4_lab.products p ON p.sku = 'SKU-300'
    LIMIT 1;
''')

run('''
    SELECT product_id, change_type, delta, balance_after, occurred_at
    FROM week4_lab.inventory_ledger
    ORDER BY occurred_at DESC, id DESC
    LIMIT 5;
''')

### Instructor Note — Trigger Validation
After execution, querying `inventory_ledger` should show a new `sale` entry with `delta = -2` and updated balance. Encourage students to match ledger balance against expectations.

## Task 4 — Exploring Transactions
Transactions ensure atomicity. Use the helper below to run experiments with commits, rollbacks, and isolation levels.

In [ ]:
from contextlib import contextmanager

@contextmanager
def transaction(isolation_level=None):
    with conn.cursor() as cur:
        cur.execute('BEGIN;')
        if isolation_level:
            cur.execute(f'SET TRANSACTION ISOLATION LEVEL {isolation_level};')
        try:
            yield cur
            cur.execute('COMMIT;')
        except Exception as exc:
            cur.execute('ROLLBACK;')
            raise exc

print('Transaction helper ready.')

### Experiment A — Rollback
Simulate a failure by inserting an invalid row and rolling back. Verify the tables remain unchanged.

In [ ]:
try:
    with transaction() as cur:
        cur.execute('SET search_path TO week4_lab;')
        cur.execute('INSERT INTO orders (customer_id, status, order_total) VALUES ((SELECT id FROM customers LIMIT 1), ''paid'', -10);')
except psycopg.errors.CheckViolation as exc:
    print('Rolled back due to:', exc.diag.message_primary)

run('SELECT COUNT(*) AS orders_count FROM week4_lab.orders;')

### Instructor Note — Rollback Experiment
Expect `Rolled back due to: new row for relation "orders" violates check constraint "orders_order_total_check"` and the final count unchanged. Use to emphasise atomicity.

### Experiment B — Isolation Levels
Open two transactions to demonstrate non-repeatable reads. Execute the cells sequentially in separate notebook kernels or terminals if possible.

In [ ]:
with transaction('REPEATABLE READ') as cur:
    cur.execute('SET search_path TO week4_lab;')
    cur.execute('SELECT order_total FROM orders LIMIT 1;')
    initial = cur.fetchone()[0]
    print('Initial total inside T1:', initial)
    print('Update the same order in another session, then continue.')
    input('Press Enter after Transaction 2 commits...')
    cur.execute('SELECT order_total FROM orders LIMIT 1;')
    after = cur.fetchone()[0]
    print('Total observed inside T1 after external commit:', after)

### Instructor Note — Isolation Walkthrough
Encourage running Transaction 2 in parallel. Under `REPEATABLE READ`, the second SELECT should return the original value, proving snapshot isolation.

> **Transaction 2 instructions** (run separately):
```
BEGIN;
SET search_path TO week4_lab;
UPDATE orders SET order_total = order_total + 10 WHERE status = 'paid' LIMIT 1;
COMMIT;
```
Expect the first transaction to maintain its snapshot under `REPEATABLE READ`.

## Midterm Proposal Reflection
Capture notes for your proposal: dataset, constraints, triggers, and transaction considerations.

In [ ]:
proposal_notes = {
    'dataset': '',
    'functional_requirements': [''],
    'integrity_rules': [''],
    'transaction_scenarios': [''],
    'open_questions': [''],
}
proposal_notes

## Cleanup (Optional)
Drop the lab schema if you need to reset the environment after capturing results.

In [ ]:
run('DROP SCHEMA IF EXISTS week4_lab CASCADE;')
print('Lab schema removed.')